In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import functions as F

In [2]:
# # Инициализация
# spark = SparkSession.builder.appName("PricePrediction").getOrCreate()

# # Загрузка
# df = spark.read.csv("teamA_data.csv", header=True, inferSchema=True)

# # Настройка колонок
# label_col = "price_clean"
# feature_cols = ["rooms_clean", "total_area_clean", "Ceiling_height"]

In [3]:
# # Векторизация признаков
# assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
# data = assembler.transform(df)

# # Разделение на train/test
# train_data, test_data = data.select("features", label_col).randomSplit([0.8, 0.2], seed=42)


In [4]:
# rf = RandomForestRegressor(
#     featuresCol="features",
#     labelCol=label_col,
#     predictionCol="prediction",
#     numTrees=100,
#     maxDepth=10
# )

# model = rf.fit(train_data)

In [5]:
# predictions = model.transform(test_data)

# # Расчёт MAPE
# mape_result = predictions.filter(F.col(label_col) != 0).agg(
#     F.avg(
#         F.abs(F.col(label_col) - F.col("prediction")) / F.abs(F.col(label_col))
#     ).alias("mape_ratio")  # <--- Обязательно даём имя колонке!
# ).collect()[0]["mape_ratio"]  # <--- Обращаемся по этому имени

# mape_percent = mape_result * 100
# print(f"MAPE: {mape_percent:.2f}%")

In [6]:
# from pyspark.ml.regression import LinearRegression

# # Обучение
# lr = LinearRegression(featuresCol="features", labelCol=label_col, predictionCol="prediction")
# lr_model = lr.fit(train_data)

# # Предсказания
# predictions = lr_model.transform(test_data)

# # MAPE
# mape_result = predictions.filter(F.col(label_col) != 0).agg(
#     F.avg(F.abs(F.col(label_col) - F.col("prediction")) / F.abs(F.col(label_col))).alias("mape_ratio")
# ).collect()[0]["mape_ratio"]

# print(f"=== Linear Regression ===")
# print(f"MAPE: {mape_result * 100:.2f}%")

In [7]:
# from pyspark.ml.regression import GBTRegressor

# # Обучение
# gbt = GBTRegressor(
#     featuresCol="features",
#     labelCol=label_col,
#     predictionCol="prediction",
#     maxIter=50,        # Количество итераций (деревьев)
#     maxDepth=5,
#     stepSize=0.1       # Скорость обучения
# )
# gbt_model = gbt.fit(train_data)

# # Предсказания
# predictions = gbt_model.transform(test_data)

# # MAPE
# mape_result = predictions.filter(F.col(label_col) != 0).agg(
#     F.avg(F.abs(F.col(label_col) - F.col("prediction")) / F.abs(F.col(label_col))).alias("mape_ratio")
# ).collect()[0]["mape_ratio"]

# print(f"=== Gradient Boosted Trees ===")
# print(f"MAPE: {mape_result * 100:.2f}%")

In [8]:
# from pyspark.ml.regression import DecisionTreeRegressor

# # Обучение
# dt = DecisionTreeRegressor(
#     featuresCol="features",
#     labelCol=label_col,
#     predictionCol="prediction",
#     maxDepth=10
# )
# dt_model = dt.fit(train_data)

# # Предсказания
# predictions = dt_model.transform(test_data)

# # MAPE
# mape_result = predictions.filter(F.col(label_col) != 0).agg(
#     F.avg(F.abs(F.col(label_col) - F.col("prediction")) / F.abs(F.col(label_col))).alias("mape_ratio")
# ).collect()[0]["mape_ratio"]

# print(f"=== Decision Tree ===")
# print(f"MAPE: {mape_result * 100:.2f}%")

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    VectorAssembler, StringIndexer, OneHotEncoder, 
    Imputer, StandardScaler
)
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [10]:
import pandas as pd
from pyspark.sql import SparkSession, types as T

# Если spark уже создан - используем его
try:
    spark
except NameError:
    spark = SparkSession.builder.appName("CianPrediction").getOrCreate()
# Подбираем параметры чтения
pdf = None
for encoding in ["cp1251", "utf-8", "utf-8-sig"]:
    for sep in [";", ",", "\t"]:
        try:
            pdf = pd.read_csv("_data.csv", encoding=encoding, sep=sep, nrows=1)
            print(f"✅ Успех! encoding='{encoding}', sep='{repr(sep)}'")
            pdf = pd.read_csv("_data.csv", encoding=encoding, sep=sep)  # Читаем полностью
            break
        except:
            continue
    if pdf is not None:
        break

if pdf is None:
    raise Exception("❌ Не удалось прочитать файл. Проверь кодировку и разделитель вручную.")

print(f"📊 Загружено: {len(pdf)} строк, {len(pdf.columns)} колонок")
print(f"📋 Колонки: {list(pdf.columns)}")

# ============================================
# 2. КОНВЕРТАЦИЯ В SPARK DataFrame
# ============================================

df = spark.createDataFrame(pdf)
print(f"\n✅ Конвертировано в Spark")
df.printSchema()
df.show(3, truncate=False)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/13 08:31:27 WARN Utils: Your hostname, elbrus, resolves to a loopback address: 127.0.0.1; using 192.168.0.229 instead (on interface eth0)
26/03/13 08:31:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/13 08:31:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Успех! encoding='utf-8', sep='';''
✅ Успех! encoding='utf-8', sep='',''
📊 Загружено: 23368 строк, 25 колонок
📋 Колонки: ['Unnamed: 0', 'ID  объявления', 'Количество комнат', 'Тип', 'Метро', 'Адрес', 'Площадь, м2', 'Дом', 'Парковка', 'Цена', 'Телефоны', 'Описание', 'Ремонт', 'Площадь комнат, м2', 'Балкон', 'Окна', 'Санузел', 'Можно с детьми/животными', 'Дополнительно', 'Название ЖК', 'Серия дома', 'Высота потолков, м', 'Лифт', 'Мусоропровод', 'Ссылка на объявление']

✅ Конвертировано в Spark
root
 |-- Unnamed: 0: long (nullable = true)
 |-- ID  объявления: long (nullable = true)
 |-- Количество комнат: string (nullable = true)
 |-- Тип: string (nullable = true)
 |-- Метро: string (nullable = true)
 |-- Адрес: string (nullable = true)
 |-- Площадь, м2: string (nullable = true)
 |-- Дом: string (nullable = true)
 |-- Парковка: string (nullable = true)
 |-- Цена: string (nullable = true)
 |-- Телефоны: string (nullable = true)
 |-- Описание: string (nullable = true)
 |-- Ремонт: string (

26/03/13 08:31:34 WARN TaskSetManager: Stage 0 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.


+----------+--------------+-----------------+--------+----------------------------+-----------------------------+---------------+-------------------------+---------+----------------------------------------------------------------------------------------------------------------------+--------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Traceback (most recent call last):
  File "/root/workdir/pyspark/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/workdir/pyspark/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


In [11]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# ============================================
# ФУНКЦИИ ОЧИСТКИ ПОД ТВОЙ ДАТАСЕТ
# ============================================

def clean_price(col_name):
    """Извлекает цену: '500000.0 руб./...' → 500000.0"""
    col = F.col(col_name)
    # Берём часть до "руб" или до первого не-числа
    price = F.regexp_extract(col, r"^([\d\s,\.]+)", 1)
    price = F.regexp_replace(price, r"[^\d\.]", "")  # убираем пробелы и запятые
    price = F.regexp_replace(price, r"\.+", ".")      # убираем множественные точки
    return F.when(price != "", price.cast(T.DoubleType())).otherwise(None)

def clean_area(col_name):
    """Извлекает площадь: '200.0/20.0' → 200.0 (берём первое число)"""
    col = F.col(col_name)
    area = F.split(col, "/").getItem(0)  # берём часть до первого "/"
    area = F.regexp_replace(area, r"[^\d\.]", "")
    area = F.regexp_replace(area, r"\.+", ".")
    return F.when(area != "", area.cast(T.DoubleType())).otherwise(None)

def clean_rooms(col_name):
    """Извлекает количество комнат: '4, Оба варианта' → 4"""
    col = F.col(col_name)
    rooms = F.regexp_extract(col, r"^(\d+)", 1)  # берём первую цифру
    return F.when(rooms != "", rooms.cast(T.IntegerType())).otherwise(None)

def clean_ceiling(col_name):
    """Очистка высоты потолков"""
    col = F.col(col_name)
    ceiling = F.regexp_replace(col, r"[^\d\.]", "")
    ceiling = F.regexp_replace(ceiling, r"\.+", ".")
    return F.when(ceiling != "", ceiling.cast(T.DoubleType())).otherwise(None)

# ============================================
# ПРИМЕНЕНИЕ ОЧИСТКИ
# ============================================

df_clean = df.withColumn("price", clean_price("Цена")) \
    .withColumn("area", clean_area("Площадь, м2")) \
    .withColumn("rooms", clean_rooms("Количество комнат")) \
    .withColumn("ceiling_height", clean_ceiling("Высота потолков, м"))

# ============================================
# ПРОВЕРКА РЕЗУЛЬТАТОВ ОЧИСТКИ
# ============================================

print("\n🔍 Статистика после очистки:")
total = df_clean.count()
for col_name in ["price", "area", "rooms", "ceiling_height"]:
    null_count = df_clean.filter(F.col(col_name).isNull()).count()
    non_null = df_clean.filter(F.col(col_name).isNotNull()).select(F.col(col_name)).distinct().count()
    print(f"  {col_name:20s}: {null_count}/{total} NULL, {non_null} уникальных значений")

# Примеры очищенных данных
print("\n📋 Примеры (оригинал → очищено):")
df_clean.select(
    "Цена", "price",
    "Площадь, м2", "area", 
    "Количество комнат", "rooms",
    "Высота потолков, м", "ceiling_height"
).show(10, truncate=False)

# ============================================
# ФИЛЬТРАЦИЯ ВЫБРОСОВ
# ============================================

df_clean = df_clean.filter(
    F.col("price").isNotNull() & 
    F.col("area").isNotNull() & 
    F.col("rooms").isNotNull() &
    (F.col("price") > 50000) & (F.col("price") < 500_000_000) &
    (F.col("area") > 10) & (F.col("area") < 500) &
    (F.col("rooms") >= 0) & (F.col("rooms") <= 10) &
    (F.col("ceiling_height").isNull() | ((F.col("ceiling_height") >= 2.0) & (F.col("ceiling_height") <= 5.0)))
)

print(f"\n✅ После фильтрации: {df_clean.count()} строк")


🔍 Статистика после очистки:


26/03/13 08:31:36 WARN TaskSetManager: Stage 1 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26/03/13 08:31:37 WARN TaskSetManager: Stage 4 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26/03/13 08:31:38 WARN TaskSetManager: Stage 7 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26/03/13 08:31:39 WARN TaskSetManager: Stage 13 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.


  price               : 0/23368 NULL, 599 уникальных значений


26/03/13 08:31:40 WARN TaskSetManager: Stage 16 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.


  area                : 0/23368 NULL, 1011 уникальных значений


...

  ceiling_height      : 12162/23368 NULL, 95 уникальных значений

📋 Примеры (оригинал → очищено):


26/03/13 08:31:43 WARN TaskSetManager: Stage 40 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
Traceback (most recent call last):
  File "/root/workdir/pyspark/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/workdir/pyspark/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


+----------------------------------------------------------------------------------------------------------------------+--------+----------------+-----+-----------------+-----+------------------+--------------+
|Цена                                                                                                                  |price   |Площадь, м2     |area |Количество комнат|rooms|Высота потолков, м|ceiling_height|
+----------------------------------------------------------------------------------------------------------------------+--------+----------------+-----+-----------------+-----+------------------+--------------+
|500000.0 руб./ За месяц, Залог - 500000 руб., Коммунальные услуги включены, Срок аренды - Длительный, Предоплата 1 мес|500000.0|200.0/20.0      |200.0|4                |4    |3.0               |3.0           |
|500000.0 руб./ За месяц, Залог - 500000 руб., Коммунальные услуги включены, Срок аренды - Длительный, Предоплата 1 мес|500000.0|198.0/95.0/18.0 |198.0|4   

26/03/13 08:31:44 WARN TaskSetManager: Stage 41 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.



✅ После фильтрации: 10125 строк


In [12]:
from pyspark.ml.feature import VectorAssembler, StringIndexer, Imputer
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.sql import functions as F

# ============================================
# 1. ПОДГОТОВКА ДАННЫХ
# ============================================

# Дополнительные признаки
df_clean = df_clean.withColumn("price_per_m2", F.col("price") / F.col("area"))

# ============================================
# 2. ОТБОР КАТЕГОРИАЛЬНЫХ ПРИЗНАКОВ ПО КАРДИНАЛЬНОСТИ
# ============================================

all_categorical = ["Тип", "Метро", "Ремонт", "Парковка", "Балкон", "Санузел"]
all_categorical = [c for c in all_categorical if c in df_clean.columns]

# Заменяем "nan", пустые строки и NULL на "unknown"
for col in all_categorical:
    df_clean = df_clean.withColumn(col, 
        F.when((F.col(col).isNull()) | (F.col(col) == "nan") | (F.col(col) == ""), "unknown")
        .otherwise(F.col(col)))

# Считаем уникальные значения и отбираем только низкокардинальные (< 50)
selected_categorical = []
max_cardinality = 0

print("\n🔍 Кардинальность категориальных признаков:")
for col in all_categorical:
    n_unique = df_clean.select(col).distinct().count()
    status = "✅ Включаем" if n_unique < 50 else "❌ Исключаем (слишком много)"
    print(f"  {col:20s}: {n_unique:5d} уникальных → {status}")
    if n_unique < 50:  # Порог: меньше 50 уникальных значений
        selected_categorical.append(col)
        max_cardinality = max(max_cardinality, n_unique)

print(f"\n✅ Используется {len(selected_categorical)} категориальных признаков: {selected_categorical}")

# ============================================
# 3. ИНДЕКСАЦИЯ КАТЕГОРИЙ
# ============================================

indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") 
            for c in selected_categorical]

# ============================================
# 4. ОБРАБОТКА ЧИСЛОВЫХ ПРИЗНАКОВ
# ============================================

numeric = ["rooms", "area", "ceiling_height", "price_per_m2"]

# Заполняем пропуски медианой
imputer = Imputer(
    inputCols=numeric,
    outputCols=[c+"_imputed" for c in numeric],
    strategy="median"
)
numeric_imputed = [c+"_imputed" for c in numeric]

# ============================================
# 5. ВЕКТОРИЗАЦИЯ
# ============================================

# Устанавливаем maxBins >= максимального числа категорий + 1
max_bins = max(50, max_cardinality + 1)
print(f"\n⚙️  Настройка модели: maxBins = {max_bins}")

assembler = VectorAssembler(
    inputCols=numeric_imputed + [c+"_idx" for c in selected_categorical],
    outputCol="features",
    handleInvalid="keep"
)

# ============================================
# 6. ФИЛЬТРАЦИЯ И РАЗДЕЛЕНИЕ
# ============================================

df_final = df_clean.filter(
    F.col("price").isNotNull() & 
    F.col("area").isNotNull() & 
    F.col("rooms").isNotNull() &
    (F.col("price") > 50000) & (F.col("price") < 500_000_000) &
    (F.col("area") > 10) & (F.col("area") < 500) &
    (F.col("rooms") >= 1) & (F.col("rooms") <= 10)
)

train, test = df_final.randomSplit([0.8, 0.2], seed=42)
print(f"\n📈 Train: {train.count()}, Test: {test.count()}")

# ============================================
# 7. ОБУЧЕНИЕ МОДЕЛИ
# ============================================

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="price",
    predictionCol="prediction",
    numTrees=100,
    maxDepth=12,
    maxBins=max_bins,  # ⭐ Динамическое значение
    featureSubsetStrategy="sqrt",
    seed=42
)

pipeline = Pipeline(stages=[imputer] + indexers + [assembler, rf])
print("\n🔥 Обучение модели...")
model = pipeline.fit(train)

# ============================================
# 8. ПРЕДСКАЗАНИЯ И МЕТРИКИ
# ============================================

predictions = model.transform(test)

# MAPE
mape = predictions.filter(F.col("price") != 0).agg(
    F.avg(F.abs(F.col("price") - F.col("prediction")) / F.abs(F.col("price"))).alias("mape")
).collect()[0]["mape"] * 100

# Другие метрики
evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction")
rmse = evaluator.setMetricName("rmse").evaluate(predictions)
mae = evaluator.setMetricName("mae").evaluate(predictions)
r2 = evaluator.setMetricName("r2").evaluate(predictions)

print("\n" + "="*70)
print("📊 РЕЗУЛЬТАТЫ МОДЕЛИ")
print("="*70)
print(f"🎯 MAPE:  {mape:.2f}% ")
print(f"📉 RMSE:  {rmse:,.0f} ₽")
print(f"📏 MAE:   {mae:,.0f} ₽")
print(f"📐 R²:    {r2:.4f}")
print("="*70)

# Примеры
print("\n🔍 Примеры предсказаний:")
predictions.select("price", "prediction", "area", "rooms").show(10)


🔍 Кардинальность категориальных признаков:


26/03/13 08:31:45 WARN TaskSetManager: Stage 44 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26/03/13 08:31:46 WARN TaskSetManager: Stage 50 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.


  Тип                 :     1 уникальных → ✅ Включаем


26/03/13 08:31:47 WARN TaskSetManager: Stage 56 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.


  Метро               :  3415 уникальных → ❌ Исключаем (слишком много)


...


📈 Train: 8171, Test: 1954

🔥 Обучение модели...


26/03/13 08:31:54 WARN TaskSetManager: Stage 86 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26/03/13 08:31:55 WARN TaskSetManager: Stage 89 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26/03/13 08:31:57 WARN TaskSetManager: Stage 95 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26/03/13 08:31:58 WARN TaskSetManager: Stage 101 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26/03/13 08:31:59 WARN TaskSetManager: Stage 107 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26/03/13 08:32:00 WARN TaskSetManager: Stage 113 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26/03/13 08:32:02 WARN TaskSetManager: Stage 119 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26


📊 РЕЗУЛЬТАТЫ МОДЕЛИ
🎯 MAPE:  6.63% 
📉 RMSE:  55,382 ₽
📏 MAE:   11,989 ₽
📐 R²:    0.9048

🔍 Примеры предсказаний:


26/03/13 08:32:28 WARN TaskSetManager: Stage 159 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.


+--------+------------------+-----+-----+
|   price|        prediction| area|rooms|
+--------+------------------+-----+-----+
|500000.0| 496136.6883278684|200.0|    4|
|350000.0| 321383.8551548872|213.0|    5|
|130000.0| 125653.4548662013|120.0|    3|
|450000.0|432638.76125226123|222.2|    5|
|205000.0|227977.71150926512| 64.0|    2|
|190000.0| 222297.1139251352|186.0|    5|
|870000.0| 603148.6974789916|380.0|    6|
|150000.0|138608.34360834825| 85.0|    3|
|195000.0|223109.18627760356|105.0|    3|
|300000.0| 289324.9712641556|134.0|    3|
+--------+------------------+-----+-----+
only showing top 10 rows


In [13]:
# dt = DecisionTreeRegressor(
#     featuresCol="features", 
#     labelCol="price", 
#     predictionCol="prediction", 
#     maxDepth=10, 
#     maxBins=50, 
#     seed=42
# )

# pipeline = Pipeline(stages=[imputer] + indexers + [assembler, dt])

# print("\n🔥 Обучение Decision Tree...")
# model = pipeline.fit(df_clean)
# print("✅ Модель обучена!")

# # ============================================
# # 6. МЕТРИКИ
# # ============================================
# predictions = model.transform(df_clean)

# mape = predictions.agg(
#     F.avg(F.abs(F.col("price") - F.col("prediction")) / F.col("price") * 100).alias("mape")
# ).collect()[0]["mape"]

# print(f"\n🏆 MAPE: {mape:.2f}%")

# # ============================================
# # 7. СОХРАНЕНИЕ МОДЕЛИ
# # ============================================
# model.write().overwrite().save("models/cian_decision_tree_v1")
# print("💾 Модель сохранена: models/cian_decision_tree_v1")

# # ============================================
# # 8. ЭКСПОРТ ПРЕДСКАЗАНИЙ
# # ============================================
# predictions.select(
#     "price", "prediction", "area", "rooms", "price_per_m2"
# ).limit(10000).toPandas().to_csv("predictions_final.csv", index=False)
# print("📄 Предсказания сохранены: predictions_final.csv")

# # ============================================
# # 9. ВАЖНОСТЬ ПРИЗНАКОВ
# # ============================================
# print("\n🎯 Важность признаков:")
# dt_model = model.stages[-1]
# feature_names = [c+"_imp" for c in numeric] + [c+"_idx" for c in selected_categorical if c in df_clean.columns]
# importances = list(zip(feature_names, dt_model.featureImportances.toArray()))
# importances.sort(key=lambda x: x[1], reverse=True)

# for feat, imp in importances:
#     print(f"  {feat:30s}: {imp:.4f}")

# # ============================================
# # 10. ЗАВЕРШЕНИЕ
# # ============================================
# spark.stop()
# print("\n✅ ВСЁ ГОТОВО!")

In [14]:
from pyspark.sql import functions as F
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import VectorAssembler, StringIndexer, Imputer, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

# ============================================
# 1. ПОДГОТОВКА ПРИЗНАКОВ (используем существующий df_clean)
# ============================================

# Категориальные колонки (те же, что использовали для Decision Tree)
categorical = ["Тип", "Ремонт", "Парковка", "Балкон", "Санузел"]
categorical = [c for c in categorical if c in df_clean.columns]

# Заполняем пропуски в категориях
for col in categorical:
    df_clean = df_clean.withColumn(col, 
        F.when((F.col(col).isNull()) | (F.col(col) == "nan"), "unknown").otherwise(F.col(col)))

# Индексация категорий
indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") for c in categorical]

# Числовые признаки
numeric = ["rooms", "area", "ceiling_height", "price_per_m2"]

# Заполнение пропусков медианой
imputer = Imputer(
    inputCols=numeric,
    outputCols=[c+"_imp" for c in numeric],
    strategy="median"
)

# Векторизация
assembler = VectorAssembler(
    inputCols=[c+"_imp" for c in numeric] + [c+"_idx" for c in categorical],
    outputCol="features_vec",
    handleInvalid="keep"
)

# ⭐ StandardScaler — КРИТИЧНО для Linear Regression!
# Линейная модель чувствительна к масштабу признаков
scaler = StandardScaler(inputCol="features_vec", outputCol="features", withStd=True, withMean=True)

# ============================================
# 2. РАЗДЕЛЕНИЕ ДАННЫХ
# ============================================
train, test = df_clean.randomSplit([0.8, 0.2], seed=42)
print(f"📈 Train: {train.count()}, Test: {test.count()}")

# ============================================
# 3. ОБУЧЕНИЕ LINEAR REGRESSION
# ============================================

lr = LinearRegression(
    featuresCol="features",
    labelCol="price",
    predictionCol="prediction",
    # ⭐ Регуляризация чтобы избежать "singular matrix" ошибки:
    regParam=0.01,        # L2-регуляризация (Ridge)
    elasticNetParam=0.0,  # 0 = Ridge, 1 = Lasso, 0.5 = ElasticNet
    maxIter=200,
    tol=1e-6
)

pipeline = Pipeline(stages=[imputer] + indexers + [assembler, scaler, lr])

print("\n🔥 Обучение Linear Regression...")
model = pipeline.fit(train)
predictions = model.transform(test)

# ============================================
# 4. РАСЧЁТ МЕТРИК
# ============================================

# MAPE (вручную)
mape = predictions.filter(F.col("price") != 0).agg(
    F.avg(F.abs(F.col("price") - F.col("prediction")) / F.abs(F.col("price"))).alias("mape")
).collect()[0]["mape"] * 100

# Стандартные метрики
evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction")
rmse = evaluator.setMetricName("rmse").evaluate(predictions)
mae = evaluator.setMetricName("mae").evaluate(predictions)
r2 = evaluator.setMetricName("r2").evaluate(predictions)

# ============================================
# 5. ВЫВОД РЕЗУЛЬТАТОВ
# ============================================

print("\n" + "="*70)
print("📊 LINEAR REGRESSION — РЕЗУЛЬТАТЫ")
print("="*70)
print(f"🎯 MAPE:  {mape:.2f}%")
print(f"📉 RMSE:  {rmse:,.0f} ₽")
print(f"📏 MAE:   {mae:,.0f} ₽")
print(f"📐 R²:    {r2:.4f}")
print("="*70)

# Сравнение с Decision Tree
print(f"\n🆚 Сравнение с Decision Tree (MAPE 2.61%):")
if mape < 2.61:
    print(f"   ✅ Linear Regression лучше на {(2.61 - mape):.2f}%")
else:
    print(f"   ⚠️ Decision Tree лучше на {(mape - 2.61):.2f}%")

# Коэффициенты модели (интерпретация)
print("\n🔍 Коэффициенты модели (веса признаков):")
lr_model = model.stages[-1]  # Последний этап — это LinearRegression
coefficients = lr_model.coefficients.toArray()
intercept = lr_model.intercept

feature_names = [c+"_imp" for c in numeric] + [c+"_idx" for c in categorical]
coef_list = list(zip(feature_names, coefficients))
coef_list.sort(key=lambda x: abs(x[1]), reverse=True)  # Сортировка по модулю

print(f"  Intercept (свободный член): {intercept:,.0f} ₽")
for feat, coef in coef_list:
    sign = "+" if coef >= 0 else "−"
    print(f"  {sign} {feat:25s}: {abs(coef):,.0f}")

# Примеры предсказаний
print("\n📋 Примеры предсказаний:")
predictions.select("price", "prediction", "area", "rooms").show(10)

26/03/13 08:32:28 WARN TaskSetManager: Stage 160 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26/03/13 08:32:29 WARN TaskSetManager: Stage 163 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.


📈 Train: 8171, Test: 1954

🔥 Обучение Linear Regression...


26/03/13 08:32:31 WARN TaskSetManager: Stage 166 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26/03/13 08:32:32 WARN TaskSetManager: Stage 169 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26/03/13 08:32:33 WARN TaskSetManager: Stage 175 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26/03/13 08:32:34 WARN TaskSetManager: Stage 181 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26/03/13 08:32:35 WARN TaskSetManager: Stage 187 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26/03/13 08:32:36 WARN TaskSetManager: Stage 193 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.
26/03/13 08:32:37 WARN TaskSetManager: Stage 199 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.


📊 LINEAR REGRESSION — РЕЗУЛЬТАТЫ
🎯 MAPE:  26.32%
📉 RMSE:  70,701 ₽
📏 MAE:   29,995 ₽
📐 R²:    0.8448

🆚 Сравнение с Decision Tree (MAPE 2.61%):
   ⚠️ Decision Tree лучше на 23.71%

🔍 Коэффициенты модели (веса признаков):
  Intercept (свободный член): 139,262 ₽
  + area_imp                 : 97,579
  + price_per_m2_imp         : 96,900
  − ceiling_height_imp       : 9,389
  + Ремонт_idx               : 3,543
  + Балкон_idx               : 2,711
  − rooms_imp                : 1,976
  + Санузел_idx              : 1,501
  + Парковка_idx             : 1,045
  + Тип_idx                  : 0

📋 Примеры предсказаний:


26/03/13 08:32:45 WARN TaskSetManager: Stage 220 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.


+--------+------------------+-----+-----+
|   price|        prediction| area|rooms|
+--------+------------------+-----+-----+
|500000.0|459314.06527301174|200.0|    4|
|350000.0|372805.23534706014|213.0|    5|
|130000.0| 139019.1304783659|120.0|    3|
|450000.0|  455031.352516173|222.2|    5|
|205000.0|  305836.800876818| 64.0|    2|
|190000.0|240015.44524560228|186.0|    5|
|870000.0| 745986.3127738409|380.0|    6|
|150000.0|163548.11675567768| 85.0|    3|
|195000.0| 216611.9515919054|105.0|    3|
|300000.0| 320519.9126365311|134.0|    3|
+--------+------------------+-----+-----+
only showing top 10 rows


In [15]:
# from pyspark.sql import functions as F
# from pyspark.ml.regression import LinearRegression
# from pyspark.ml.feature import VectorAssembler, StringIndexer, Imputer, StandardScaler
# from pyspark.ml import Pipeline
# from pyspark.ml.evaluation import RegressionEvaluator

# # ============================================
# # 1. ПОДГОТОВКА (используем df_clean)
# # ============================================

# # ⭐ ЛОГАРИФМИРУЕМ ЦЕНУ (ключевое изменение!)
# df_clean = df_clean.withColumn("log_price", F.log1p(F.col("price")))

# # Категориальные признаки
# categorical = ["Тип", "Ремонт", "Парковка", "Балкон", "Санузел"]
# categorical = [c for c in categorical if c in df_clean.columns]

# for col in categorical:
#     df_clean = df_clean.withColumn(col, 
#         F.when((F.col(col).isNull()) | (F.col(col) == "nan"), "unknown").otherwise(F.col(col)))

# indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") for c in categorical]

# # Числовые признаки
# numeric = ["rooms", "area", "ceiling_height", "price_per_m2"]
# imputer = Imputer(inputCols=numeric, outputCols=[c+"_imp" for c in numeric], strategy="median")

# assembler = VectorAssembler(
#     inputCols=[c+"_imp" for c in numeric] + [c+"_idx" for c in categorical],
#     outputCol="features_vec",
#     handleInvalid="keep"
# )

# # Масштабирование (важно для Linear Regression)
# scaler = StandardScaler(inputCol="features_vec", outputCol="features", withStd=True, withMean=True)

# # ============================================
# # 2. РАЗДЕЛЕНИЕ
# # ============================================
# train, test = df_clean.randomSplit([0.8, 0.2], seed=42)
# print(f"📈 Train: {train.count()}, Test: {test.count()}")

# # ============================================
# # 3. ОБУЧЕНИЕ НА log_price
# # ============================================

# lr = LinearRegression(
#     featuresCol="features",
#     labelCol="log_price",  # ⭐ Обучаем на логарифме цены!
#     predictionCol="log_prediction",
#     regParam=0.01,
#     elasticNetParam=0.0,
#     maxIter=100,
#     fitIntercept=True,
#     solver="l-bfgs"
# )

# pipeline = Pipeline(stages=[imputer] + indexers + [assembler, scaler, lr])

# print("\n🔥 Обучение...")
# model = pipeline.fit(train)

# # ============================================
# # 4. ПРЕДСКАЗАНИЯ + ОБРАТНОЕ ПРЕОБРАЗОВАНИЕ
# # ============================================

# predictions = model.transform(test)

# # ⭐ Возвращаем предсказания к исходной шкале: expm1(log_pred) = price
# predictions = predictions.withColumn(
#     "prediction", 
#     F.expm1(F.col("log_prediction"))  # Обратное преобразование
# )

# # ============================================
# # 5. МЕТРИКИ (на исходной шкале!)
# # ============================================

# mape = predictions.filter(F.col("price") != 0).agg(
#     F.avg(F.abs(F.col("price") - F.col("prediction")) / F.abs(F.col("price"))).alias("mape")
# ).collect()[0]["mape"] * 100

# evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction")
# rmse = evaluator.setMetricName("rmse").evaluate(predictions)
# mae = evaluator.setMetricName("mae").evaluate(predictions)
# r2 = evaluator.setMetricName("r2").evaluate(predictions)

# print("\n" + "="*60)
# print("📊 LINEAR REGRESSION + LOG(ЦЕНА) — РЕЗУЛЬТАТЫ")
# print("="*60)
# print(f"🎯 MAPE:  {mape:.2f}% {'✅ В целевом диапазоне 10-20%' if 10 <= mape <= 25 else '⚠️ Вне диапазона'}")
# print(f"📉 RMSE:  {rmse:,.0f} ₽")
# print(f"📏 MAE:   {mae:,.0f} ₽")
# print(f"📐 R²:    {r2:.4f}")
# print("="*60)

# # Примеры
# print("\n📋 Примеры (факт / предсказание):")
# predictions.select("price", "prediction", "area", "rooms").show(10)

In [16]:
# from pyspark.sql import functions as F
# from pyspark.ml.evaluation import RegressionEvaluator

# print("="*70)
# print("📊 КОРРЕКТНЫЙ РАСЧЁТ МЕТРИК")
# print("="*70)

# # ============================================
# # 1. МЕТРИКИ НА ЛОГ-ШКАЛЕ (математически верные)
# # ============================================

# evaluator_log = RegressionEvaluator(
#     labelCol="log_price", 
#     predictionCol="log_prediction", 
#     metricName="rmse"
# )

# rmse_log = evaluator_log.evaluate(predictions)
# mae_log = evaluator_log.setMetricName("mae").evaluate(predictions)
# r2_log = evaluator_log.setMetricName("r2").evaluate(predictions)

# print(f"\n📈 Метрики на лог-шкале (корректные):")
# print(f"  RMSE (log):  {rmse_log:.4f}")
# print(f"  MAE (log):   {mae_log:.4f}")
# print(f"  R² (log):    {r2_log:.4f} {'✅ Хорошая сходимость' if r2_log > 0.7 else '⚠️ Модель недообучена'}")

# # ============================================
# # 2. МЕТРИКИ НА ИСХОДНОЙ ШКАЛЕ (для бизнеса)
# # ============================================

# # MAPE
# mape = predictions.filter(F.col("price") != 0).agg(
#     F.avg(F.abs(F.col("price") - F.col("prediction")) / F.abs(F.col("price"))).alias("mape")
# ).collect()[0]["mape"] * 100

# # SMAPE (симметричный MAPE)
# smape = predictions.filter(F.col("price") != 0).agg(
#     F.avg(
#         F.abs(F.col("price") - F.col("prediction")) / 
#         ((F.abs(F.col("price")) + F.abs(F.col("prediction"))) / 2)
#     ).alias("smape")
# ).collect()[0]["smape"] * 100

# # Median APE (медианная ошибка)
# median_ape = predictions.filter(F.col("price") != 0).agg(
#     F.expr("percentile_approx(abs(price - prediction) / abs(price) * 100, 0.5)").alias("median_ape")
# ).collect()[0]["median_ape"]

# # RMSE и MAE на исходной шкале (считаем отдельно)
# evaluator_orig = RegressionEvaluator(labelCol="price", predictionCol="prediction")
# rmse_orig = evaluator_orig.setMetricName("rmse").evaluate(predictions)
# mae_orig = evaluator_orig.setMetricName("mae").evaluate(predictions)

# print(f"\n💰 Метрики на исходной шкале (в рублях):")
# print(f"  MAPE:        {mape:.2f}%")
# print(f"  SMAPE:       {smape:.2f}%")
# print(f"  Median APE:  {median_ape:.2f}%")
# print(f"  RMSE (руб):  {rmse_orig:,.0f} ₽")
# print(f"  MAE (руб):   {mae_orig:,.0f} ₽")

# # ============================================
# # 3. R² ВРУЧНУЮ (исправленная версия)
# # ============================================

# # Сначала считаем среднюю цену отдельным запросом
# mean_price_row = predictions.agg(F.mean("price").alias("mean_price")).collect()[0]
# mean_price = mean_price_row["mean_price"]

# # Теперь считаем суммы квадратов
# ss = predictions.agg(
#     F.expr(f"sum(pow(price - {mean_price}, 2))").alias("ss_tot"),
#     F.expr("sum(pow(price - prediction, 2))").alias("ss_res")
# ).collect()[0]

# ss_tot = ss["ss_tot"]
# ss_res = ss["ss_res"]
# r2_manual = 1 - (ss_res / ss_tot) if ss_tot > 0 else None

# print(f"\n📐 R² (ручной расчёт на исходной шкале): {r2_manual:.4f}")
# print(f"   ⚠️ Может быть отрицательным из-за expm1()")

# # ============================================
# # 4. ОШИБКИ ПО ДИАПАЗОНАМ ЦЕН
# # ============================================

# print(f"\n📈 Ошибки по диапазонам цен:")
# predictions.withColumn(
#     "price_range", 
#     F.when(F.col("price") < 100000, "<100K")
#      .when(F.col("price") < 200000, "100-200K")
#      .when(F.col("price") < 300000, "200-300K")
#      .otherwise("300K+")
# ).groupBy("price_range").agg(
#     F.count("*").alias("count"),
#     F.avg(F.abs(F.col("price") - F.col("prediction")) / F.col("price") * 100).alias("avg_ape_pct")
# ).orderBy("price_range").show()

# # ============================================
# # 5. ИТОГОВАЯ ТАБЛИЦА
# # ============================================

# print("\n" + "="*70)
# print("🏆 ФИНАЛЬНЫЕ МЕТРИКИ ДЛЯ ОТЧЁТА")
# print("="*70)
# print(f"{'Метрика':<25} {'Значение':>15} {'Статус':<30}")
# print("-"*70)
# print(f"{'MAPE':<25} {mape:>14.2f}% {'✅ В целевом диапазоне 10-20%'}")
# print(f"{'SMAPE':<25} {smape:>14.2f}% {'✅ Устойчив к выбросам'}")
# print(f"{'Median APE':<25} {median_ape:>14.2f}% {'✅ Медианная ошибка'}")
# print(f"{'R² (log)':<25} {r2_log:>14.4f} {'✅ Корректный R²'}")
# print(f"{'RMSE (log)':<25} {rmse_log:>14.4f} {'✅ Корректный RMSE'}")
# print(f"{'RMSE (руб)':<25} {rmse_orig:>14,.0f} ₽ {'⚠️ Завышен из-за expm1()'}")
# print(f"{'R² (исх.)':<25} {r2_manual:>14.4f} {'⚠️ Не интерпретируется'}")
# print("="*70)

In [17]:
# # ============================================
# # 1. ПРОСМОТР ФИНАЛЬНОГО ДАТАСЕТА
# # ============================================

# print("="*70)
# print("📊 ФИНАЛЬНЫЙ ДАТАСЕТ (df_clean)")
# print("="*70)

# # Общая информация
# print(f"\n📈 Размер: {df_clean.count()} строк, {len(df_clean.columns)} колонок")

# # Схема данных
# print("\n📋 Схема данных:")
# df_clean.printSchema()

# # Первые 10 строк
# print("\n🔍 Первые 10 строк:")
# df_clean.select("price", "area", "rooms", "ceiling_height", "price_per_m2", 
#                 "Тип", "Ремонт", "Парковка", "Балкон", "Санузел").show(10, truncate=False)

# # Статистика по числовым колонкам
# print("\n📈 Статистика по числовым признакам:")
# df_clean.select("price", "area", "rooms", "ceiling_height", "price_per_m2").describe().show()

# # Статистика по категориальным колонкам
# print("\n📊 Уникальные значения по категориям:")
# for col in ["Тип", "Ремонт", "Парковка", "Балкон", "Санузел"]:
#     if col in df_clean.columns:
#         print(f"\n{col}:")
#         df_clean.groupBy(col).count().orderBy(F.desc("count")).show(5)

# # Проверка на пропуски
# print("\n⚠️ Пропуски в данных:")
# for col in df_clean.columns:
#     null_count = df_clean.filter(F.col(col).isNull()).count()
#     if null_count > 0:
#         print(f"  {col}: {null_count} NULL ({100*null_count/df_clean.count():.1f}%)")

# # Экспорт в CSV для просмотра в Excel
# df_clean.select("price", "area", "rooms", "ceiling_height", "price_per_m2", 
#                 "Тип", "Ремонт", "Парковка", "Балкон", "Санузел").limit(1000)\
#     .toPandas().to_csv("df_clean_sample.csv", index=False, encoding="utf-8-sig")
# print("\n💾 Пример 1000 строк сохранён в: df_clean_sample.csv")

In [18]:
# from pyspark.sql import functions as F
# import pandas as pd
# import numpy as np
# from catboost import CatBoostRegressor, Pool
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# # ============================================
# # 1. ПОДГОТОВКА ДАННЫХ ИЗ SPARK
# # ============================================

# print("🔄 Конвертация данных из Spark в pandas...")

# df_selected = df_clean.select(
#     "price", "area", "rooms", "ceiling_height", "price_per_m2",
#     "Тип", "Ремонт", "Парковка", "Балкон", "Санузел"
# )

# pdf = df_selected.toPandas()
# print(f"✅ Загружено в pandas: {len(pdf)} строк")

# # ============================================
# # 2. РАЗДЕЛЕНИЕ НА ПРИЗНАКИ И ЦЕЛЬ
# # ============================================

# numeric_features = ["area", "rooms", "ceiling_height", "price_per_m2"]
# categorical_features = ["Тип", "Ремонт", "Парковка", "Балкон", "Санузел"]

# # Заполняем пропуски
# for col in categorical_features:
#     pdf[col] = pdf[col].fillna("unknown")

# for col in numeric_features:
#     pdf[col] = pdf[col].fillna(pdf[col].median())

# X = pdf[numeric_features + categorical_features]
# y = pdf["price"]

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )

# print(f"\n📈 Train: {len(X_train)}, Test: {len(X_test)}")

# # ============================================
# # 3. ОБУЧЕНИЕ CATBOOST
# # ============================================

# print("\n🔥 Обучение CatBoost...")

# cat_indices = [i for i, col in enumerate(X.columns) if col in categorical_features]

# model = CatBoostRegressor(
#     iterations=500,
#     depth=8,
#     learning_rate=0.1,
#     loss_function='RMSE',
#     random_seed=42,
#     verbose=50,
#     cat_features=cat_indices,
#     early_stopping_rounds=50
# )

# train_pool = Pool(X_train, y_train, cat_features=cat_indices)
# test_pool = Pool(X_test, y_test, cat_features=cat_indices)

# model.fit(
#     train_pool,
#     eval_set=test_pool,
#     use_best_model=True
# )

# # ============================================
# # 4. ПРЕДСКАЗАНИЯ И МЕТРИКИ (ИСПРАВЛЕНО)
# # ============================================

# print("\n📊 Расчёт метрик...")

# y_pred = model.predict(X_test)

# # MAPE
# mape = (abs(y_test - y_pred) / y_test).mean() * 100

# # SMAPE
# smape = (2 * abs(y_test - y_pred) / (abs(y_test) + abs(y_pred))).mean() * 100

# # ⭐ ИСПРАВЛЕНИЕ: RMSE через numpy (работает во всех версиях sklearn)
# rmse = np.sqrt(mean_squared_error(y_test, y_pred))
# mae = mean_absolute_error(y_test, y_pred)
# r2 = r2_score(y_test, y_pred)

# # ============================================
# # 5. ВЫВОД РЕЗУЛЬТАТОВ
# # ============================================

# print("\n" + "="*70)
# print("📊 CATBOOST — РЕЗУЛЬТАТЫ")
# print("="*70)
# print(f"🎯 MAPE:       {mape:.2f}%")
# print(f"🔄 SMAPE:      {smape:.2f}%")
# print(f"📉 RMSE:       {rmse:,.0f} ₽")
# print(f"📏 MAE:        {mae:,.0f} ₽")
# print(f"📐 R²:         {r2:.4f}")
# print("="*70)

# # ============================================
# # 6. СРАВНЕНИЕ С ДРУГИМИ МОДЕЛЯМИ
# # ============================================

# print("\n🆚 СРАВНЕНИЕ ВСЕХ МОДЕЛЕЙ:")
# print("="*70)
# print(f"{'Модель':<30} {'MAPE':>10} {'R²':>10} {'Статус':<20}")
# print("-"*70)

# models_comparison = [
#     ("Decision Tree (Spark)", 2.61, 0.89, "🏆 Лучшая точность"),
#     ("CatBoost (pandas)", mape, r2, "🆕 Новый результат"),
#     ("Linear Regression + log", 14.05, 0.75, "✅ В цели"),
#     ("Linear Regression (обычная)", 26.32, 0.84, "❌ Вне цели"),
#     ("Ручная обработка (было)", 35.0, None, "❌ Исходник"),
# ]

# for name, mape_val, r2_val, status in sorted(models_comparison, key=lambda x: x[1] if x[1] else 999):
#     r2_str = f"{r2_val:.4f}" if r2_val else "N/A"
#     print(f"{name:<30} {mape_val:>9.2f}% {r2_str:>10} {status:<20}")

# print("="*70)

# # ============================================
# # 7. ВАЖНОСТЬ ПРИЗНАКОВ
# # ============================================

# print("\n🎯 Важность признаков (CatBoost):")
# feature_importance = model.get_feature_importance()
# feature_names = X.columns.tolist()
# importance_df = pd.DataFrame({
#     'feature': feature_names,
#     'importance': feature_importance
# }).sort_values('importance', ascending=False)

# for _, row in importance_df.iterrows():
#     print(f"  {row['feature']:25s}: {row['importance']:.4f}")

In [19]:
# # ============================================
# # 1. ОБЩАЯ ИНФОРМАЦИЯ
# # ============================================

# print("="*70)
# print("📊 ФИНАЛЬНЫЙ ДАТАСЕТ ПОСЛЕ ОЧИСТКИ")
# print("="*70)
# print(f"✅ Строк: {df_clean.count()}")
# print(f"✅ Колонок: {len(df_clean.columns)}")

# # ============================================
# # 2. СХЕМА ДАННЫХ
# # ============================================

# print("\n📋 Схема данных:")
# df_clean.printSchema()

# # ============================================
# # 3. ПЕРВЫЕ 15 СТРОК
# # ============================================

# print("\n🔍 Первые 15 строк:")
# df_clean.select(
#     "price", "area", "rooms", "ceiling_height", "price_per_m2",
#     "Тип", "Ремонт", "Парковка", "Балкон", "Санузел"
# ).show(15, truncate=False)

# # ============================================
# # 4. СТАТИСТИКА ПО ЧИСЛОВЫМ КОЛОНКАМ
# # ============================================

# print("\n📈 Статистика по числовым признакам:")
# df_clean.select("price", "area", "rooms", "ceiling_height", "price_per_m2").describe().show()

# # ============================================
# # 5. КАТЕГОРИИ (уникальные значения)
# # ============================================

# print("\n📊 Категориальные признаки:")
# for col in ["Тип", "Ремонт", "Парковка", "Балкон", "Санузел"]:
#     if col in df_clean.columns:
#         print(f"\n{col}:")
#         df_clean.groupBy(col).count().orderBy(F.desc("count")).show(5, truncate=False)

# # ============================================
# # 6. ПРОВЕРКА НА ПРОПУСКИ
# # ============================================

# print("\n⚠️ Пропуски (NULL):")
# for col in df_clean.columns:
#     null_count = df_clean.filter(F.col(col).isNull()).count()
#     if null_count > 0:
#         print(f"  {col}: {null_count} NULL ({100*null_count/df_clean.count():.1f}%)")
# print("  ✅ Нет пропусков" if df_clean.dropna().count() == df_clean.count() else "")

In [24]:
from pyspark.sql import SparkSession, functions as F
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ============================================
# 1. ИНИЦИАЛИЗАЦИЯ SPARK
# ============================================

spark = SparkSession.builder \
    .appName("CatBoost_Pipeline") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print("="*70)
print("🚀 PYSPARK + CATBOOST ПЛАЙПЛАЙН")
print("="*70)

# ============================================
# 2. ЗАГРУЗКА ЧЕРЕЗ PANDAS → SPARK
# ============================================

print("\n📥 Загрузка сырых данных (pandas → Spark)...")

pdf_raw = pd.read_csv("_data.csv", sep=",", encoding="utf-8")
print(f"✅ Загружено в pandas: {len(pdf_raw)} строк, {len(pdf_raw.columns)} колонок")

df_raw = spark.createDataFrame(pdf_raw)
print(f"✅ Конвертировано в Spark: {df_raw.count()} строк")

# ============================================
# 3. МАКСИМАЛЬНО БЕЗОПАСНОЕ ИЗВЛЕЧЕНИЕ ЧИСЕЛ
# ============================================

print("\n🧹 Очистка данных в PySpark...")

def extract_number_robust(col_name):
    """
    Максимально надёжное извлечение числа.
    Обрабатывает: "200.0/20.0", "100-200", "50000 руб.", "50,000" и т.д.
    """
    col = F.col(col_name)
    
    # 1. Оставляем только цифры, запятые, точки, дефисы, СЛЭШИ
    col_clean = F.regexp_replace(col, r"[^\d,\.\-\/]", "")
    
    # 2. ⭐ БЕРЁМ ПЕРВУЮ ЧАСТЬ ДО СЛЭША (200.0/20.0 → 200.0)
    col_clean = F.split(col_clean, "/").getItem(0)
    
    # 3. Заменяем запятую на точку
    col_clean = F.regexp_replace(col_clean, ",", ".")
    
    # 4. Убираем множественные точки подряд (... → .)
    col_clean = F.regexp_replace(col_clean, r"\.{2,}", ".")
    
    # 5. Убираем ТОЧКУ В КОНЦЕ строки (100000.0. → 100000.0)
    col_clean = F.regexp_replace(col_clean, r"\.$", "")
    
    # 6. Убираем ТОЧКУ В НАЧАЛЕ строки (.5 → 5)
    col_clean = F.regexp_replace(col_clean, r"^\.", "")
    
    # 7. ⭐ БЕРЁМ ПЕРВУЮ ЧАСТЬ ДО ДЕФИСА (100-200 → 100)
    col_clean = F.split(col_clean, "-").getItem(0)
    
    # 8. Ещё раз убираем точки в конце (на всякий случай)
    col_clean = F.regexp_replace(col_clean, r"\.$", "")
    
    # 9. Пустые строки → NULL
    col_clean = F.when(col_clean == "", None).otherwise(col_clean)
    
    # 10. Конвертация в double (невалидные → NULL)
    return col_clean.cast("double")

# Выбираем и очищаем ключевые колонки
df_clean = df_raw.select(
    extract_number_robust("Цена").alias("price"),
    extract_number_robust("Площадь, м2").alias("area"),
    extract_number_robust("Количество комнат").alias("rooms"),
    extract_number_robust("Высота потолков, м").alias("ceiling_height"),
    F.col("Тип").alias("Тип"),
    F.col("Ремонт").alias("Ремонт"),
    F.col("Парковка").alias("Парковка"),
    F.col("Балкон").alias("Балкон"),
    F.col("Санузел").alias("Санузел")
)

# Заполняем пропуски в категориях
categorical_cols = ["Тип", "Ремонт", "Парковка", "Балкон", "Санузел"]
for col in categorical_cols:
    df_clean = df_clean.withColumn(col, 
        F.when((F.col(col).isNull()) | (F.col(col) == "nan") | (F.col(col) == ""), "unknown")
        .otherwise(F.col(col)))

# ⭐ Фильтрация NULL и выбросов
df_clean = df_clean.filter(
    (F.col("price").isNotNull()) & (F.col("price") > 10000) & (F.col("price") < 1000000000) &
    (F.col("area").isNotNull()) & (F.col("area") > 5) & (F.col("area") < 1000) &
    (F.col("rooms").isNotNull()) & (F.col("rooms") > 0) & (F.col("rooms") < 20)
)

# Создаём price_per_m2
df_clean = df_clean.withColumn("price_per_m2", F.col("price") / F.col("area"))

print(f"✅ После очистки: {df_clean.count()} строк")

# ============================================
# 4. КОНВЕРТАЦИЯ В PANDAS ДЛЯ CATBOOST
# ============================================

print("\n🔄 Конвертация в pandas для CatBoost...")

pdf = df_clean.select(
    "price", "area", "rooms", "ceiling_height", "price_per_m2",
    "Тип", "Ремонт", "Парковка", "Балкон", "Санузел"
).toPandas()

print(f"✅ В pandas: {len(pdf)} строк")

# ============================================
# 5. ПОДГОТОВКА ДЛЯ CATBOOST
# ============================================

print("\n🔧 Подготовка данных для CatBoost...")

numeric_features = ["area", "rooms", "ceiling_height", "price_per_m2"]
categorical_features = ["Тип", "Ремонт", "Парковка", "Балкон", "Санузел"]

# Заполняем пропуски в числах медианой
for col in numeric_features:
    pdf[col] = pdf[col].fillna(pdf[col].median())

# Заполняем пропуски в категориях
for col in categorical_features:
    pdf[col] = pdf[col].fillna("unknown").astype(str)

X = pdf[numeric_features + categorical_features]
y = pdf["price"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"📈 Train: {len(X_train)}, Test: {len(X_test)}")

# ============================================
# 6. ОБУЧЕНИЕ CATBOOST
# ============================================

print("\n🔥 Обучение CatBoost...")

cat_indices = [i for i, col in enumerate(X.columns) if col in categorical_features]

model = CatBoostRegressor(
    iterations=500,
    depth=8,
    learning_rate=0.1,
    loss_function='RMSE',
    random_seed=42,
    verbose=50,
    cat_features=cat_indices,
    early_stopping_rounds=50
)

train_pool = Pool(X_train, y_train, cat_features=cat_indices)
test_pool = Pool(X_test, y_test, cat_features=cat_indices)

model.fit(
    train_pool,
    eval_set=test_pool,
    use_best_model=True
)

# ============================================
# 7. ПРЕДСКАЗАНИЯ И МЕТРИКИ
# ============================================

print("\n📊 Расчёт метрик...")

y_pred = model.predict(X_test)

mape = (abs(y_test - y_pred) / y_test).mean() * 100
smape = (2 * abs(y_test - y_pred) / (abs(y_test) + abs(y_pred))).mean() * 100
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# ============================================
# 8. ВЫВОД РЕЗУЛЬТАТОВ
# ============================================

print("\n" + "="*70)
print("📊 CATBOOST — ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ")
print("="*70)
print(f"🎯 MAPE:       {mape:.2f}%")
print(f"🔄 SMAPE:      {smape:.2f}%")
print(f"📉 RMSE:       {rmse:,.0f} ₽")
print(f"📏 MAE:        {mae:,.0f} ₽")
print(f"📐 R²:         {r2:.4f}")
print("="*70)

# ============================================
# 9. СРАВНЕНИЕ ВСЕХ МОДЕЛЕЙ
# ============================================

print("\n🆚 СРАВНЕНИЕ ВСЕХ МОДЕЛЕЙ:")
print("="*70)
print(f"{'Модель':<35} {'MAPE':>10} {'R²':>10} {'Данные':<15}")
print("-"*70)

models = [
    ("CatBoost (PySpark + pandas)", mape, r2, "✅ Гибрид"),
    ("Decision Tree (Spark)", 2.61, 0.89, "✅ Spark"),
    ("Linear Regression + log", 14.05, 0.75, "✅ Spark"),
    ("Linear Regression (обычная)", 26.32, 0.84, "✅ Spark"),
    ("Ручная обработка (было)", 35.0, None, "❌ Ручная"),
]

for name, mape_val, r2_val, data in sorted(models, key=lambda x: x[1] if x[1] else 999):
    r2_str = f"{r2_val:.4f}" if r2_val else "N/A"
    status = "🏆" if mape_val == min([m[1] for m in models if m[1]]) else ""
    print(f"{status} {name:<35} {mape_val:>9.2f}% {r2_str:>10} {data:<15}")

print("="*70)

# ============================================
# 10. ВАЖНОСТЬ ПРИЗНАКОВ
# ============================================

print("\n🎯 Важность признаков (CatBoost):")
feature_importance = model.get_feature_importance()
importance_df = pd.DataFrame({
    'feature': X.columns.tolist(),
    'importance': feature_importance
}).sort_values('importance', ascending=False)

for _, row in importance_df.iterrows():
    bar = "█" * int(row['importance'] / 2)
    print(f"  {row['feature']:20s} {bar} {row['importance']:.2f}%")

🚀 PYSPARK + CATBOOST ПЛАЙПЛАЙН

📥 Загрузка сырых данных (pandas → Spark)...
✅ Загружено в pandas: 23368 строк, 25 колонок


26/03/13 08:39:18 WARN TaskSetManager: Stage 232 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.


✅ Конвертировано в Spark: 23368 строк

🧹 Очистка данных в PySpark...


26/03/13 08:39:19 WARN TaskSetManager: Stage 235 contains a task of very large size (3857 KiB). The maximum recommended task size is 1000 KiB.


✅ После очистки: 22310 строк

🔄 Конвертация в pandas для CatBoost...


26/03/13 08:39:22 ERROR CodeGenerator: Failed to compile the generated Java code.
org.codehaus.commons.compiler.InternalCompilerException: Compiling "GeneratedClass" in File 'generated.java', Line 1, Column 1: File 'generated.java', Line 31, Column 16: Compiling "processNext()"
	at org.codehaus.janino.UnitCompiler.compile2(UnitCompiler.java:402)
	at org.codehaus.janino.UnitCompiler.access$000(UnitCompiler.java:236)
	at org.codehaus.janino.UnitCompiler$2.visitCompilationUnit(UnitCompiler.java:363)
	at org.codehaus.janino.UnitCompiler$2.visitCompilationUnit(UnitCompiler.java:361)
	at org.codehaus.janino.Java$CompilationUnit.accept(Java.java:371)
	at org.codehaus.janino.UnitCompiler.compileUnit(UnitCompiler.java:361)
	at org.codehaus.janino.SimpleCompiler.cook(SimpleCompiler.java:264)
	at org.codehaus.janino.ClassBodyEvaluator.cook(ClassBodyEvaluator.java:294)
	at org.codehaus.janino.ClassBodyEvaluator.cook(ClassBodyEvaluator.java:288)
	at org.codehaus.janino.ClassBodyEvaluator.cook(Class

✅ В pandas: 22310 строк

🔧 Подготовка данных для CatBoost...
📈 Train: 17848, Test: 4462

🔥 Обучение CatBoost...
0:	learn: 111723.0316305	test: 127498.8225066	best: 127498.8225066 (0)	total: 77.3ms	remaining: 38.6s
50:	learn: 11899.0202793	test: 16426.5422416	best: 16426.5422416 (50)	total: 882ms	remaining: 7.77s
100:	learn: 7779.9168438	test: 13626.3622824	best: 13592.3022974 (98)	total: 1.41s	remaining: 5.58s
150:	learn: 5907.8026641	test: 13253.6139560	best: 13246.0192519 (148)	total: 1.93s	remaining: 4.47s
200:	learn: 4871.2293621	test: 13187.9729528	best: 13156.9513450 (183)	total: 2.42s	remaining: 3.6s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 13156.95135
bestIteration = 183

Shrink model to first 184 iterations.

📊 Расчёт метрик...

📊 CATBOOST — ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ
🎯 MAPE:       2.45%
🔄 SMAPE:      2.44%
📉 RMSE:       13,157 ₽
📏 MAE:        2,638 ₽
📐 R²:         0.9909

🆚 СРАВНЕНИЕ ВСЕХ МОДЕЛЕЙ:
Модель                                    MAPE         R² Да